# D3 ε sweep — full privacy-utility Pareto under generic + adaptive attack

Sweeps DP-SGD target ε ∈ {0.5, 1.0, 3.0, 10.0, ∞} on EEGNet over PhysioNet's 104 subjects. For each ε we run both attacks (logreg probe and encoder fine-tune adaptive) so the privacy-utility frontier covers both ends of the attacker spectrum.

Output JSON drives a Pareto curve showing task accuracy vs. re-ID under both attacker strengths across the formal-DP frontier.

**Runtime → L4 GPU.** Expected wall: ~140-170 min (5 DP trainings × ~18 min + 5 fine-tunes + 5 logreg attacks).

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.29_d3_eps_sweep --all

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
RESULTS = 'results/29_d3_eps_sweep.json'
TAG = 'd3_eps_sweep'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.29_d3_eps_sweep',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing; re-run experiment cell.')
print('--- BEGIN 29_d3_eps_sweep.json ---')
print(open(RESULTS).read())
print('--- END 29_d3_eps_sweep.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')